# Cleaning Validation & Data Quality Audit — CatalogIQ

**Ticket criteria.** Compare raw versus cleaned data for: row counts, quarantine
counts and reasons, missingness, target distributions, rare-class retention,
datatype changes, feature normalization, malformed labels, structural
corruption, and unexpected information loss. Particular attention to whether
legitimate rare classes or valid product records were accidentally removed.

Every section prints a **VERDICT** line — PASS, WARN, FAIL or BLOCKED — with the
evidence above it. The last section collects them into one table and writes it to
`interim/cleaning_audit_summary.csv`.

**BLOCKED is not PASS.** A check that could not run is recorded as blocked, with
the reason. That distinction is the point: five of these criteria compare raw
against cleaned, and reporting them as passing when no comparison happened would
be worse than reporting nothing.

**What this notebook needs.**

| | Where | Produced by |
|---|---|---|
| Raw | `provided/Selfcare_Training_data (1).csv` | the PO's ZIP |
| Cleaned | `interim/` or `processed/` | the cleaning pipeline |
| `summary.json` | shipped with the handoff | the cleaning pipeline |

**Full mode** needs raw and cleaned, and runs all ten criteria.

**Partial mode** runs when the raw file is absent but the cleaned file and
`summary.json` are present. Row counts reconcile from the handoff manifest, the
checks that need only the cleaned file still run, and the five that need raw
report BLOCKED.

The setup cell says which mode it is in and why.

**The bias of this audit.** A cleaning step that removes rows always looks like
it worked: the data gets tidier. The question this notebook asks is the opposite
one — what left, and should it have. So every check is framed as loss detection,
and a silent drop is treated as a failure even when the cleaned data looks
better for it.

**Related work.** The malformed-label findings come from the relationships
research (§2.1): each target carries exactly 3 malformed values, all beginning
with whitespace. Section 7 checks whether cleaning handled those as expected.
Rare-class retention (section 6) feeds the evaluation strategy directly — a
class dropped in cleaning changes the macro-F1 denominator.

## 0. Setup

Finds the project root by walking up to the folder holding `provided/`, then
looks for the cleaned dataset in `interim/` and `processed/`.

Both load as string. In the raw target dataset the classification columns are
empty and pandas infers `float64`; comparing dtypes between a string load and an
inferred load would report changes that are an artifact of loading, not of
cleaning. Section 4 reads the dtypes separately, with inference on, to report
real datatype changes.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 170)

# Set these to override auto-detection.
RAW_PATH     = None
CLEANED_PATH = None

RAW_NAME = "Selfcare_Training_data (1).csv"
TARGETS  = ["Mnfr", "Brand", "Platform", "Segment", "Sub-Segment", "TargetAgeGroup"]

# Candidate keys for matching a raw row to a cleaned row, best first.
# source_row is the pipeline's provenance key: a 1-based index into the raw file
# (summary.json records "source_row_base": 1). It survives the feature export,
# which drops JoiningKey/Sku/Upc, so it is the only key that works end to end.
KEY_CANDIDATES = ["source_row", "JoiningKey", "Sku", "Upc"]


# The repo ships its own path constants in src/catalogiq/paths.py. Use them when
# importable so this notebook cannot drift from the rest of the project.
CATALOGIQ_PATHS = None
try:
    import sys
    for _p in (Path.cwd(), *Path.cwd().parents):
        if (_p / "src" / "catalogiq" / "paths.py").is_file():
            sys.path.insert(0, str(_p / "src"))
            break
    from catalogiq import paths as CATALOGIQ_PATHS   # noqa: E402
    print("using catalogiq.paths:", CATALOGIQ_PATHS.ROOT_PATH)
except Exception as _e:
    print("catalogiq.paths not importable (", type(_e).__name__, ") - falling back")


def find_project_root(start="."):
    if CATALOGIQ_PATHS is not None:
        return CATALOGIQ_PATHS.ROOT_PATH
    p = Path(start).resolve()
    for _ in range(8):
        if (p / "data" / "provided").is_dir() or (p / "provided").is_dir()                 or (p / ".git").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return None


ROOT = find_project_root()
# This repo keeps data under data/ ; older layouts used the repo root.
DATA = (CATALOGIQ_PATHS.DATA_PATH if CATALOGIQ_PATHS is not None
        else ((ROOT / "data") if (ROOT and (ROOT / "data").is_dir()) else ROOT))
print("project root:", ROOT or "not found")
print("data dir    :", DATA or "not found")

if RAW_PATH is None and DATA and (DATA / "provided" / RAW_NAME).is_file():
    RAW_PATH = str(DATA / "provided" / RAW_NAME)

# The cleaned file: any CSV in interim/ or processed/ carrying the targets.
if CLEANED_PATH is None and DATA:
    cands = []
    for sub in ("processed", "interim"):
        d = DATA / sub
        if d.is_dir():
            cands += sorted(d.glob("*.csv"))
            # the pipeline writes dated run directories under processed/
            cands += sorted(d.glob("*/*.csv"))
    scored = []
    for f in cands:
        try:
            cols = set(pd.read_csv(f, nrows=0).columns)
        except Exception:
            continue
        present = [c for c in TARGETS if c in cols]
        if len(present) < 4:
            continue
        # Prefer the LABELLED file: the unlabelled target set has the same
        # columns but they are empty, and comparing against it is meaningless.
        head = pd.read_csv(f, dtype=str, usecols=present, nrows=5000)
        labelled = bool(head.notna().any(axis=None))
        n = sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) - 1
        scored.append((labelled, n, f))
    if scored:
        scored.sort(key=lambda t: (not t[0], -t[1]))   # labelled first, then largest
        CLEANED_PATH = str(scored[0][2])
        if len(scored) > 1:
            print()
            print("cleaned candidates (labelled first):")
            for lab, n, f in scored:
                print(f"   {n:>7,} rows  {'labelled' if lab else 'unlabelled'}  {f}")

print("raw    :", RAW_PATH or "NOT FOUND")
print("cleaned:", CLEANED_PATH or "NOT FOUND")
print()

# The cleaning handoff's own metadata, if the preparer shipped it.
import json as _json
SUMMARY = None
if ROOT:
    for cand in list(ROOT.rglob("summary.json"))[:8]:
        try:
            SUMMARY = _json.loads(cand.read_text())
            print()
            print("handoff summary found:", cand)
            break
        except Exception:
            pass

MODE = "full" if RAW_PATH else ("partial" if (CLEANED_PATH and SUMMARY) else "blocked")
print()
print("audit mode:", MODE.upper())
if MODE == "partial":
    print("  Raw data is absent, so row-level comparison is impossible.")
    print("  Checks that need only the cleaned file still run. Checks that need")
    print("  the raw file are recorded as BLOCKED, not as PASS.")

missing = []
if MODE == "blocked" and not RAW_PATH:
    missing.append(
        "RAW: expected data/provided/" + RAW_NAME + "\n"
        "     The PO's ZIPs are git-ignored, so cloning the repo does not include\n"
        "     them. Extract them into provided/ with filenames unchanged."
    )
if MODE == "blocked" and not CLEANED_PATH:
    missing.append(
        "CLEANED: expected a CSV in interim/ or processed/ with the target columns.\n"
        "     This audit compares raw against cleaned, so it needs the output of\n"
        "     the cleaning pipeline, not just the raw file. If cleaning has not run\n"
        "     yet, or writes somewhere else, point CLEANED_PATH at it."
    )
if missing:
    print("Cannot run the audit yet.\n")
    for m in missing:
        print(" - " + m + "\n")
    raise SystemExit("Provide the missing dataset(s), then re-run.")

cleaned = pd.read_csv(CLEANED_PATH, dtype=str, keep_default_na=True)
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=True) if RAW_PATH else None

print()
print("cleaned", cleaned.shape)
if raw is not None:
    print("raw    ", raw.shape)
    print("raw columns not in cleaned:", sorted(set(raw.columns)     - set(cleaned.columns)))
    print("cleaned columns not in raw:", sorted(set(cleaned.columns) - set(raw.columns)))

### 0.1 Verdict recorder

Each section appends to `VERDICTS`. Section 11 prints the table.

In [ ]:
VERDICTS = []


def verdict(criterion, status, detail):
    """status: PASS, WARN, FAIL or BLOCKED (needs data we do not have)."""
    VERDICTS.append({"criterion": criterion, "status": status, "detail": detail})
    bar = {"PASS": "", "WARN": "!  ", "FAIL": "!! ", "BLOCKED": "-- "}[status]
    print(f"\nVERDICT [{criterion}]: {bar}{status}")
    for line in str(detail).split("\n"):
        print("   " + line)

### 0.2 Choosing a row key

To tell which records disappeared, raw and cleaned rows have to be matched. A
row-count difference alone says how many left, not which ones or whether they
should have.

Preference order is `JoiningKey`, `Sku`, `Upc`. If none of them is usable, the
notebook falls back to a content hash over the shared columns, which still
detects removals but cannot distinguish a removed row from a heavily edited one.

In [ ]:
KEY = None
# Columns the two files have in common - used by several sections below.
shared = ([c for c in raw.columns if c in cleaned.columns and not c.startswith("_")]
          if raw is not None
          else [c for c in cleaned.columns if not c.startswith("_")])

if raw is None:
    KEY = next((k for k in KEY_CANDIDATES if k in cleaned.columns), None)
    print("partial mode - nothing to match against; key noted for reference:", KEY)
else:
    # The pipeline writes source_row on its outputs but it is implicit in the raw
    # file: row N of raw is source_row N (1-based). Reconstruct it so the two
    # frames can be matched even after the export drops the business keys.
    if "source_row" in cleaned.columns and "source_row" not in raw.columns:
        raw["source_row"] = (raw.index + 1).astype(str)
        print("reconstructed source_row on raw (1-based row index)")

    for k in KEY_CANDIDATES:
        if k in raw.columns and k in cleaned.columns:
            miss = raw[k].isna().mean()
            dupe = raw[k].duplicated().mean()
            print(f"{k:12s} missing {miss:6.2%}  duplicated {dupe:6.2%}")
            if KEY is None and miss < 0.01:
                KEY = k

    if KEY:
        print(f"\nusing key: {KEY}")
        if raw[KEY].duplicated().any():
            print("  note: key is not unique - rows are compared as multisets")
    else:
        print("\nno usable key column - falling back to a content hash over shared columns")
        raw["_rowhash"]     = pd.util.hash_pandas_object(raw[shared].fillna(""),     index=False)
        cleaned["_rowhash"] = pd.util.hash_pandas_object(cleaned[shared].fillna(""), index=False)
        KEY = "_rowhash"
        print("  caution: an edited row looks like a removed row plus a new one")

## 1. Row counts

The headline number, and the weakest one on its own. It is here because the
ticket asks for it and because it bounds everything else: if no rows were
removed, sections 2 and 6 cannot find losses.

In [ ]:
removed, added = set(), set()

if raw is None:
    # Reconcile from the handoff metadata instead.
    ds = (SUMMARY or {}).get("datasets", {})
    key = "training" if "training" in ds else next(iter(ds), None)
    meta = ds.get(key, {}) if key else {}
    n_in   = meta.get("input")
    n_out  = meta.get("cleaned")
    quar_n = meta.get("quarantine")
    hold_n = meta.get("review_hold")
    print(f"from summary.json, dataset '{key}':")
    print(f"  input        {n_in:>8,}" if n_in else "  input        unknown")
    print(f"  cleaned      {n_out:>8,}" if n_out else "  cleaned      unknown")
    print(f"  quarantine   {quar_n:>8,}" if quar_n is not None else "")
    print(f"  review_hold  {hold_n:>8,}" if hold_n is not None else "")
    print()
    print(f"  rows on disk {len(cleaned):>8,}")

    if n_out and len(cleaned) != n_out:
        verdict("row counts", "FAIL",
                f"summary.json says {n_out:,} cleaned rows but the file has "
                f"{len(cleaned):,}. The handoff does not match its own manifest.")
    elif n_in and n_out and quar_n is not None and hold_n is not None:
        removed_n = n_in - n_out
        accounted = quar_n + hold_n
        print(f"  removed      {removed_n:>8,}")
        print(f"  accounted    {accounted:>8,}  (quarantine + review_hold)")
        if removed_n == accounted:
            verdict("row counts", "PASS",
                    f"{removed_n:,} rows removed ({removed_n/n_in:.2%}); "
                    f"reconciles exactly to {quar_n:,} quarantined + {hold_n:,} "
                    f"held for review. File matches its manifest.")
        else:
            verdict("row counts", "FAIL",
                    f"{removed_n:,} removed but {accounted:,} accounted for - "
                    f"{removed_n - accounted:,} unexplained.")
    else:
        verdict("row counts", "BLOCKED",
                "summary.json does not carry the counts needed to reconcile.")
else:
    n_raw, n_clean = len(raw), len(cleaned)
    delta = n_clean - n_raw
    pct = delta / n_raw if n_raw else 0
    print(f"raw      {n_raw:>8,}")
    print(f"cleaned  {n_clean:>8,}")
    print(f"change   {delta:>+8,}  ({pct:+.3%})")
    raw_keys   = set(raw[KEY].dropna())
    clean_keys = set(cleaned[KEY].dropna())
    removed    = raw_keys - clean_keys
    added      = clean_keys - raw_keys
    print()
    print(f"keys in raw only (removed): {len(removed):,}")
    print(f"keys in cleaned only (added): {len(added):,}")

if raw is None:
    pass
elif delta == 0 and not removed:
    verdict("row counts", "PASS", "No rows removed or added.")
elif added:
    verdict("row counts", "FAIL",
            f"{len(added):,} rows exist in cleaned but not in raw. Cleaning should "
            f"not invent records; check for a bad join or a duplicated key.")
elif abs(pct) > 0.05:
    verdict("row counts", "WARN",
            f"{-delta:,} rows removed ({-pct:.2%}). Above 5% - sections 2 and 6 "
            f"decide whether that is justified.")
else:
    verdict("row counts", "PASS",
            f"{-delta:,} rows removed ({-pct:.2%}); within an expected range for "
            f"malformed-row removal. Section 2 must still account for them.")

## 2. Quarantine counts and reasons

Every removed row needs a reason. A removal with no recorded reason is the
failure this section exists to catch: it is indistinguishable from a bug.

The notebook looks for a quarantine or rejects file in `interim/`. If the
pipeline does not produce one, that is itself a finding — removals are
unauditable without it.

In [ ]:
quar = None
if ROOT:
    # row_decisions.csv is the pipeline's full decision table: one row per source
    # row with the rules that fired. It is the real answer to "why was this
    # removed", so look for it before the partition files.
    pats = ["*row_decisions*", "*quarantin*", "*reject*", "*dropped*",
            "*excluded*", "*invalid*"]
    hits = []
    for sub in ("processed", "interim"):
        d = DATA / sub
        if d.is_dir():
            for p in pats:
                hits += sorted(d.glob(p + ".csv")) + sorted(d.glob("*/" + p + ".csv"))
    if hits:
        print("quarantine file(s) found:")
        for h in hits:
            print("   ", h)
        quar = pd.read_csv(hits[0], dtype=str)
        print()
        print("shape:", quar.shape)
        print("columns:", list(quar.columns))

if quar is None:
    print("No quarantine file found in data/interim/ or data/processed/.")
    print("Looked for names containing: quarantine, reject, dropped, excluded, invalid")

print()
if raw is None:
    ds = (SUMMARY or {}).get("datasets", {})
    key = "training" if "training" in ds else next(iter(ds), None)
    meta = ds.get(key, {}) if key else {}
    q, h = meta.get("quarantine"), meta.get("review_hold")
    if quar is None and q is not None:
        verdict("quarantine counts and reasons", "FAIL",
                f"summary.json reports {q:,} quarantined and {h:,} held for "
                f"review, but neither file was shipped with the handoff - the "
                f"README says they are 'kept separately by the preparer'. The "
                f"counts are known; the per-row reasons are not, so no removal "
                f"can be checked against the rule that caused it. Request the "
                f"quarantine and review-hold files.")
    elif quar is None:
        verdict("quarantine counts and reasons", "BLOCKED",
                "No quarantine file and no counts in summary.json.")
    else:
        verdict("quarantine counts and reasons", "PASS",
                "Quarantine file shipped with the handoff.")
elif quar is None and removed:
    verdict("quarantine counts and reasons", "FAIL",
            f"{len(removed):,} rows were removed and no quarantine file records "
            f"why. Removals are unauditable. The pipeline should write rejected "
            f"rows with a reason column.")
elif quar is None:
    verdict("quarantine counts and reasons", "PASS",
            "No rows removed, so no quarantine expected.")
else:
    reason_col = next((c for c in quar.columns
                       if "reason" in c.lower() or "rule" in c.lower()
                       or "error" in c.lower()), None)
    # A full decision table carries one row per source row, most of them kept.
    # Count only the rows that actually name a reason.
    if reason_col and quar[reason_col].notna().sum() < len(quar):
        quar = quar[quar[reason_col].notna()]
        print(f"decision table: {len(quar):,} row(s) carry a removal reason")
    # The decision table covers every dataset the pipeline processed. Keep only
    # the one this audit is comparing, or the counts will not reconcile.
    if "dataset" in quar.columns and "dataset" in cleaned.columns:
        mine = set(cleaned["dataset"].dropna().unique())
        before = len(quar)
        quar = quar[quar["dataset"].isin(mine)]
        if len(quar) != before:
            print(f"restricted to dataset {sorted(mine)}: "
                  f"{len(quar):,} of {before:,} rows")
    if reason_col:
        counts = quar[reason_col].value_counts(dropna=False)
        print("reasons:")
        print(counts.to_string())
        print()
    covered = len(quar)
    gap = len(removed) - covered
    if reason_col is None:
        verdict("quarantine counts and reasons", "WARN",
                f"Quarantine file has {covered:,} rows but no reason column. "
                f"Counts reconcile but the 'why' is missing.")
    elif gap == 0:
        verdict("quarantine counts and reasons", "PASS",
                f"All {covered:,} removed rows are accounted for, with reasons.")
    else:
        verdict("quarantine counts and reasons", "FAIL",
                f"{len(removed):,} rows removed but {covered:,} quarantined - "
                f"{gap:,} unexplained.")

## 3. Missingness

Cleaning should not increase missingness. If a column became emptier, a
transform is failing silently and writing nulls where it could not parse.

In [ ]:
if raw is None:
    print(cleaned[shared].isna().mean().mul(100).round(2).sort_values(ascending=False).head(20).to_string())
    verdict("missingness", "BLOCKED",
            "Needs the raw file to compare per-column missing rates. The cleaned file's own missing rates are reported above.")
else:
    m = pd.DataFrame({
        "raw_missing":     raw[shared].isna().mean(),
        "cleaned_missing": cleaned[shared].isna().mean(),
    })
    m["change_pp"] = (m["cleaned_missing"] - m["raw_missing"]) * 100
    m = m.sort_values("change_pp", ascending=False)

    show = m.copy()
    show[["raw_missing", "cleaned_missing"]] *= 100      # change_pp already is pp
    print(show.round(2).to_string())

    worse = m[m["change_pp"] > 0.5]
    print()
    if worse.empty:
        verdict("missingness", "PASS",
                "No column became more than 0.5pp emptier after cleaning.")
    else:
        verdict("missingness", "FAIL",
                "Columns that lost data in cleaning:\n" +
                "\n".join(f"  {c}: +{r['change_pp']:.2f}pp missing"
                          for c, r in worse.iterrows()) +
                "\nA transform is writing nulls where it cannot parse.")

## 4. Datatype changes

Read with inference on, so this reports what a downstream consumer would
actually get. A target column that reads as `float64` is the §2.4 trap: it means
the column is entirely empty, and a join against labels will fail silently.

In [ ]:
clean_dt = pd.read_csv(CLEANED_PATH, nrows=20000).dtypes
raw_dt   = pd.read_csv(RAW_PATH, nrows=20000).dtypes if RAW_PATH else clean_dt

dt = pd.DataFrame({"raw": raw_dt, "cleaned": clean_dt}).dropna()
dt["changed"] = dt["raw"].astype(str) != dt["cleaned"].astype(str)
changed = dt[dt["changed"]]

print(dt.to_string())
print()

bad = [c for c in TARGETS
       if c in clean_dt.index and str(clean_dt[c]).startswith("float")]
if bad:
    verdict("datatype changes", "FAIL",
            f"Target column(s) read as float: {bad}. That means they are entirely "
            f"empty. Force string on load, or a label join fails silently (§2.4).")
elif raw is None:
    verdict("datatype changes", "PASS",
            "No target column reads as float, so the empty-column trap is not "
            "present. The raw-vs-cleaned dtype comparison needs the raw file.")
elif changed.empty:
    verdict("datatype changes", "PASS", "No dtype changed between raw and cleaned.")
else:
    verdict("datatype changes", "WARN",
            "Dtypes changed:\n" +
            "\n".join(f"  {c}: {r['raw']} -> {r['cleaned']}"
                      for c, r in changed.iterrows()) +
            "\nExpected for parsed numerics; check nothing became object by accident.")

## 5. Target distributions

Class shares should be close to unchanged. A share that moves materially means
cleaning removed rows unevenly across classes, which biases every model trained
on the result — and does it invisibly, because the cleaned data still looks
well-formed.

In [ ]:
if raw is None:
    verdict("target distributions", "BLOCKED",
            "Needs the raw file. Class shares cannot shift against a baseline that is not present.")
else:
    for c in TARGETS:
        if c not in raw.columns or c not in cleaned.columns:
            continue
        r = raw[c].value_counts(dropna=True)
        k = cleaned[c].value_counts(dropna=True)
        comp = pd.DataFrame({"raw_n": r, "cleaned_n": k}).fillna(0).astype(int)
        comp["delta"] = comp["cleaned_n"] - comp["raw_n"]
        comp["raw_share"]     = comp["raw_n"]     / comp["raw_n"].sum()
        comp["cleaned_share"] = comp["cleaned_n"] / max(comp["cleaned_n"].sum(), 1)
        comp["share_pp"] = (comp["cleaned_share"] - comp["raw_share"]) * 100
        comp = comp.sort_values("raw_n", ascending=False)
        print(f"--- {c} ---")
        print(comp[["raw_n", "cleaned_n", "delta", "share_pp"]].head(15).round(3).to_string())
        print()

    shifts = []
    for c in TARGETS:
        if c not in raw.columns or c not in cleaned.columns:
            continue
        r = raw[c].value_counts(normalize=True)
        k = cleaned[c].value_counts(normalize=True)
        both = r.index.union(k.index)
        d = (k.reindex(both).fillna(0) - r.reindex(both).fillna(0)).abs().max() * 100
        shifts.append((c, d))

    worst = max(shifts, key=lambda t: t[1]) if shifts else ("n/a", 0)
    if worst[1] < 1:
        verdict("target distributions", "PASS",
                f"Largest class-share move is {worst[1]:.2f}pp ({worst[0]}).")
    elif worst[1] < 5:
        verdict("target distributions", "WARN",
                f"{worst[0]} shifted {worst[1]:.2f}pp. Check section 6 for which class.")
    else:
        verdict("target distributions", "FAIL",
                f"{worst[0]} shifted {worst[1]:.2f}pp. Cleaning removed rows unevenly "
                f"across classes; models trained on this inherit the bias.")

## 6. Rare-class retention

**The check the ticket singles out, and the one most likely to find something.**

Three outcomes have to be told apart, and a row count alone cannot do it:

| What happened | Looks like | Is it a problem |
|---|---|---|
| Class merged into a near-duplicate label | class vanished | No — intended |
| Class deleted with its rows | class vanished | **Yes** |
| Class shrank but survived | fewer rows | Depends |

The merge case matters because of the near-duplicates found in the relationships
research: `Cold / Flu` (4 rows in the profile) against `Cold/Flu`, and two
similar pairs. If cleaning folded the small one into the large one, the large
one's count should have grown by the small one's. If it did not, those rows were
deleted, and a 4-row class became a 0-row class quietly.

In the 2026-10-02 handoff all three small variants are at 0 rows and each
survivor is *smaller* than in the profile, which is consistent with overall row
removal but does not demonstrate a merge. Settling it needs the raw file.

In [ ]:
if raw is None:
    verdict("rare-class retention", "BLOCKED",
            "Needs the raw file. This is the check the ticket singles out, so it is the strongest reason to obtain the raw CSVs rather than work from the handoff.")
else:
    def norm_label(s):
        return (str(s).lower()
                .replace(" / ", "/").replace("/ ", "/").replace(" /", "/")
                .strip())


    RARE_THRESHOLD = 10
    findings, hard_fail = [], False

    # The pipeline records every label rewrite in label_changes.csv. A label that
    # vanished because its rows were rewritten to another label is a documented
    # relabel, not a loss - and the surviving label's net count can still fall if
    # it lost rows to quarantine at the same time, so counting alone misreads it.
    relabels = {}
    if ROOT:
        for f in list(DATA.rglob("label_changes.csv"))[:3] if DATA else []:
            try:
                lc = pd.read_csv(f, dtype=str)
                for _, r in lc.iterrows():
                    relabels.setdefault((r["column"], r["old_value"]), []).append(
                        (r["new_value"], r.get("rule", "")))
                print(f"label_changes.csv: {len(lc)} recorded relabel(s) from {f.name}")
            except Exception:
                pass

    for c in TARGETS:
        if c not in raw.columns or c not in cleaned.columns:
            continue
        r = raw[c].value_counts()
        k = cleaned[c].value_counts()

        # Malformed labels are meant to be removed - section 7 owns them. Counting
        # them here would bury a real deletion under expected ones.
        vanished = [lbl for lbl in r.index
                    if lbl not in k.index and not str(lbl).strip() != str(lbl)]
        if not vanished:
            continue

        print(f"--- {c}: {len(vanished)} label(s) present in raw, absent in cleaned ---")
        for lbl in vanished:
            n = int(r[lbl])
            key = norm_label(lbl)
            # did a surviving label absorb these rows?
            absorbed_by = None
            for other in k.index:
                if other != lbl and norm_label(other) == key:
                    grew = int(k[other]) - int(r.get(other, 0))
                    absorbed_by = (other, grew)
                    break

            rec = relabels.get((c, lbl))
            if rec:
                dest = sorted({d for d, _ in rec})
                rules = sorted({ru for _, ru in rec if ru})
                print(f"  RELABELLED {lbl!r} ({n}) -> {', '.join(dest)}"
                      f"  [{', '.join(rules) or 'recorded'}]  {len(rec)} row(s)")
                if len(rec) < n:
                    findings.append(
                        f"{c}: {lbl!r} had {n} rows but only {len(rec)} relabels "
                        f"recorded; {n - len(rec)} unaccounted for")
                    hard_fail = True
                continue

            if absorbed_by and absorbed_by[1] <= 0:
                absorbed_by = None          # a sibling label exists but absorbed nothing
            if absorbed_by and absorbed_by[1] >= n:
                print(f"  MERGED   {lbl!r} ({n}) -> {absorbed_by[0]!r} (+{absorbed_by[1]})")
            elif absorbed_by:
                short = n - absorbed_by[1]
                print(f"  PARTIAL  {lbl!r} ({n}) -> {absorbed_by[0]!r} "
                      f"(+{absorbed_by[1]}); {short} rows unaccounted for")
                findings.append(f"{c}: {lbl!r} merged but {short} rows lost")
                hard_fail = True
            else:
                tag = "RARE " if n < RARE_THRESHOLD else ""
                print(f"  DELETED  {tag}{lbl!r} ({n} rows) - no label absorbed these")
                findings.append(f"{c}: {lbl!r} deleted with {n} rows")
                hard_fail = True
        print()

    # classes that survived but shrank sharply
    print("--- surviving classes that lost more than 20% of their rows ---")
    shrunk = []
    for c in TARGETS:
        if c not in raw.columns or c not in cleaned.columns:
            continue
        r = raw[c].value_counts()
        k = cleaned[c].value_counts()
        for lbl in k.index:
            if lbl in r.index and r[lbl] > 0:
                loss = 1 - k[lbl] / r[lbl]
                if loss > 0.20:
                    shrunk.append((c, lbl, int(r[lbl]), int(k[lbl]), loss))
    if shrunk:
        for c, lbl, a, b, loss in sorted(shrunk, key=lambda t: -t[4]):
            print(f"  {c:16s} {lbl!r}: {a} -> {b}  (-{loss:.0%})")
    else:
        print("  none")

    print()
    if hard_fail:
        verdict("rare-class retention", "FAIL",
                "Classes lost rows that no other label absorbed:\n" +
                "\n".join("  " + f for f in findings) +
                "\nEach one needs a recorded decision before the data is used. A "
                "deleted class also changes the macro-F1 denominator in the "
                "evaluation strategy.")
    elif shrunk:
        verdict("rare-class retention", "WARN",
                f"No class was deleted, but {len(shrunk)} class(es) lost over 20% of "
                f"their rows. Confirm those removals were intended.")
    else:
        verdict("rare-class retention", "PASS",
                "No class disappeared and no class lost more than 20% of its rows.")

## 7. Malformed labels

The relationships research established that each target carries exactly 3
malformed values, all starting with whitespace (`" tea"`, `" lactose"`,
`" Keto certified"`), and that they are shifted rows rather than real labels.

Cleaning should have removed or repaired all of them. Any that survive will be
treated as real classes by every downstream step.

In [ ]:
def ws_mask(s):
    return s.notna() & s.astype(str).str.match(r"^\s|\s$")


rows = []
for c in TARGETS:
    if c not in cleaned.columns:
        continue
    rb = int(ws_mask(raw[c]).sum()) if raw is not None and c in raw.columns else 0
    cb = int(ws_mask(cleaned[c]).sum())
    rows.append({"target": c, "raw_malformed": rb, "cleaned_malformed": cb})
    if rb:
        print(f"{c}: raw {sorted(raw.loc[ws_mask(raw[c]), c].unique())}")
    if cb:
        print(f"{c}: SURVIVED {sorted(cleaned.loc[ws_mask(cleaned[c]), c].unique())}")

mal = pd.DataFrame(rows)
print()
print(mal.to_string(index=False))
print()

left = int(mal["cleaned_malformed"].sum()) if len(mal) else 0
found = int(mal["raw_malformed"].sum()) if len(mal) else 0
if left == 0 and found > 0:
    verdict("malformed labels", "PASS",
            f"All {found} malformed values across the targets were handled.")
elif left == 0:
    verdict("malformed labels", "PASS", "No malformed values in raw or cleaned.")
else:
    verdict("malformed labels", "FAIL",
            f"{left} malformed value(s) survived cleaning. Downstream steps will "
            f"treat them as real classes and they will inflate the class count of "
            f"every target they appear in.")

## 8. Feature normalization

What cleaning actually changed in the feature columns, for rows that survived.
Reported per column as the share of matched rows whose value changed, with
examples — a normalization that touches 90% of a column is a different thing
from one that touches 0.1%, and neither is visible in a row count.

In [ ]:
if raw is None:
    verdict("feature normalization", "BLOCKED",
            "Needs the raw file to diff values row by row. The handoff README lists the transforms it applied; those are claims, not verification.")
else:
    FEATURES = [c for c in shared
                    if c not in TARGETS and c != KEY and not c.startswith("_")]

    if KEY in cleaned.columns and not raw[KEY].duplicated().any():
        j = raw.set_index(KEY)[FEATURES].join(
            cleaned.set_index(KEY)[FEATURES], lsuffix="_raw", rsuffix="_cln", how="inner")
        print(f"matched rows: {len(j):,}")
        print()
        out = []
        for c in FEATURES:
            a, b = j[c + "_raw"].fillna(""), j[c + "_cln"].fillna("")
            diff = (a != b)
            if diff.any():
                ex = j.loc[diff, [c + "_raw", c + "_cln"]].head(3)
                out.append({"column": c, "changed_rows": int(diff.sum()),
                            "share": diff.mean()})
                print(f"--- {c}: {diff.sum():,} changed ({diff.mean():.1%}) ---")
                print(ex.to_string())
                print()
        norm = pd.DataFrame(out).sort_values("share", ascending=False) if out else pd.DataFrame()
        if len(norm):
            print(norm.to_string(index=False))
        verdict("feature normalization", "PASS",
                f"{len(out)} feature column(s) were modified. Listed above with "
                f"examples; confirm each transform was intended.")
    else:
        print("Key is not unique - skipping per-row comparison.")
        verdict("feature normalization", "WARN",
                "Could not compare row by row without a unique key. Report the "
                "transforms from the pipeline code instead.")

## 9. Structural corruption

Shifted rows are the corruption the relationships research identified: a value
from one column lands in another. The whitespace prefix in section 7 is the
symptom; this section looks for the cause and for anything similar that cleaning
may have left behind.

Two cheap signals: a target column holding a value that belongs to a different
target's vocabulary, and a row whose field count differs from the header.

In [ ]:
src = raw if raw is not None else cleaned
vocab = {c: set(src[c].dropna().astype(str).str.strip().unique())
         for c in TARGETS if c in src.columns}

cross = []
for c in TARGETS:
    if c not in cleaned.columns:
        continue
    vals = set(cleaned[c].dropna().astype(str).str.strip().unique())
    for other, ov in vocab.items():
        if other == c:
            continue
        overlap = vals & ov - vocab.get(c, set())
        if overlap:
            cross.append((c, other, sorted(overlap)[:5], len(overlap)))

if cross:
    print("values appearing in a target they do not belong to:")
    for c, other, ex, n in cross:
        print(f"  {c} holds {n} value(s) from {other}'s vocabulary: {ex}")
else:
    print("no cross-column vocabulary leakage detected")

# field-count check against the header
bad_rows = 0
with open(CLEANED_PATH, encoding="utf-8", errors="ignore") as f:
    header = f.readline().count(",")
    for i, line in enumerate(f):
        if abs(line.count(",") - header) > 0 and '"' not in line:
            bad_rows += 1
print()
print(f"rows whose unquoted comma count differs from the header: {bad_rows}")

print()
if not cross and bad_rows == 0:
    verdict("structural corruption", "PASS",
            "No cross-column value leakage and no ragged rows in the cleaned file.")
elif cross:
    verdict("structural corruption", "FAIL",
            f"{len(cross)} column pair(s) share values that should belong to only "
            f"one of them. This is the shifted-row signature and it survived "
            f"cleaning.")
else:
    verdict("structural corruption", "WARN",
            f"{bad_rows} row(s) have an unexpected field count. May be quoting; "
            f"inspect before trusting the parse.")

## 10. Unexpected information loss

Everything that left the dataset, in one place, so the reviewer can judge the
total rather than each check in isolation. Cleaning is allowed to remove things;
it is not allowed to remove things nobody decided to remove.

In [ ]:
if raw is None:
    verdict("unexpected information loss", "BLOCKED",
            "Needs the raw file to list dropped columns and removed rows. summary.json gives totals only, not which records left.")
else:
    print("=== columns dropped ===")
    dropped_cols = sorted(set(raw.columns) - set(cleaned.columns) - {"_rowhash"})
    print(dropped_cols or "none")

    print()
    print("=== rows removed, by whether a target label was present ===")
    if removed:
        gone = raw[raw[KEY].isin(removed)]
        summary = []
        for c in TARGETS:
            if c in gone.columns:
                summary.append({"target": c,
                                "removed_rows_with_label": int(gone[c].notna().sum())})
        print(pd.DataFrame(summary).to_string(index=False))
        print()
        print(f"removed rows carrying at least one label: "
              f"{int(gone[[c for c in TARGETS if c in gone.columns]].notna().any(axis=1).sum()):,}"
              f" of {len(gone):,}")
        print()
        print("sample of removed rows:")
        show = [c for c in ["ProductName", "Brand", "Segment", "Sub-Segment"] if c in gone.columns]
        print(gone[show].head(10).to_string(index=False) if show else gone.head(10).to_string())
    else:
        print("none")

    print()
    labelled_loss = 0
    if removed:
        gone = raw[raw[KEY].isin(removed)]
        cols = [c for c in TARGETS if c in gone.columns]
        labelled_loss = int(gone[cols].notna().any(axis=1).sum())

    if not dropped_cols and not removed:
        verdict("unexpected information loss", "PASS", "Nothing was removed.")
    elif labelled_loss and quar is None:
        verdict("unexpected information loss", "FAIL",
                f"{labelled_loss:,} labelled rows were removed with no quarantine "
                f"record. Labelled rows are the scarce resource in this project - "
                f"Platform has only ~1,100 of them.")
    elif dropped_cols:
        verdict("unexpected information loss", "WARN",
                f"Columns dropped: {dropped_cols}. Confirm each was intended and is "
                f"not needed as a feature or as classification evidence.")
    else:
        verdict("unexpected information loss", "PASS",
                f"{len(removed):,} unlabelled rows removed, all quarantined.")

## 11. Audit summary

The deliverable. Any FAIL means the cleaned dataset should not be used for
modelling until the item is resolved or an explicit decision is recorded in
`docs/decisions.md`.

In [ ]:
summary = pd.DataFrame(VERDICTS)
if len(summary):
    order = {"FAIL": 0, "WARN": 1, "BLOCKED": 2, "PASS": 3}
    summary = summary.sort_values("status", key=lambda s: s.map(order))
    print(summary[["criterion", "status"]].to_string(index=False))
    print()
    counts = summary["status"].value_counts()
    print(f"FAIL {counts.get('FAIL', 0)}   "
          f"WARN {counts.get('WARN', 0)}   "
          f"BLOCKED {counts.get('BLOCKED', 0)}   "
          f"PASS {counts.get('PASS', 0)}")
    print()
    for _, r in summary.iterrows():
        if r["status"] != "PASS":
            print(f"[{r['status']}] {r['criterion']}")
            for line in str(r["detail"]).split("\n"):
                print("    " + line)
            print()

    if DATA:
        (DATA / "interim").mkdir(parents=True, exist_ok=True)
        out = DATA / "interim" / "cleaning_audit_summary.csv"
        summary.to_csv(out, index=False)
        print("written to", out)
else:
    print("no verdicts recorded - run the notebook from the top")

### What to do with this

1. **Any FAIL blocks use of the cleaned data.** Either fix the pipeline or record
   an explicit decision in `docs/decisions.md` saying the loss is accepted and why.
2. **Rare-class deletions go to the evaluation strategy too.** A class that no
   longer exists changes the macro-F1 denominator and the support tiers in §3.2
   and §3.5 of that document.
3. **Malformed labels that survived (section 7) invalidate the relationships
   work**, which assumes cleaning removed them before any crosstab runs (§2.1).
4. **A missing quarantine file is a pipeline gap, not an audit finding.** Removals
   that nobody can explain are the thing this audit exists to surface.